In [21]:
import pandas as pd

df = pd.read_csv("../data/hospital_data.csv")

print("Original Shape:", df.shape)

df.head()

Original Shape: (12000, 12)


,Patient_ID,Appointment_Date,Appointment_Time,Arrival_Time,Department,Doctor_ID,Patient_Type,Queue_Length,Doctors_Available,Appointments_Count,Consultation_Duration_Min,Waiting_Time_Minutes
0,P05732,2026-01-01,08:00,07:54,ENT,DOC013,New,11,3,20,21,20
1,P02024,2026-01-01,09:10,09:33,General Medicine,DOC016,Returning,8,1,31,12,28
2,P00180,2026-01-01,09:30,09:46,Orthopedics,DOC016,New,7,1,27,25,34
3,P03650,2026-01-01,10:00,09:54,General Medicine,DOC003,New,10,2,25,16,23
4,P03679,2026-01-01,10:20,10:17,Gynecology,DOC013,Returning,12,3,29,22,15


In [22]:
df["Appointment_Date"] = pd.to_datetime(
    df["Appointment_Date"]
)

print(df["Appointment_Date"].dtype)

datetime64[us]


In [23]:
df["Appointment_Month"] = df["Appointment_Date"].dt.month

df["Appointment_Day"] = df["Appointment_Date"].dt.day

df["Day_of_Week"] = df["Appointment_Date"].dt.dayofweek

df["Is_Weekend"] = (
    df["Day_of_Week"] >= 5
).astype(int)

In [24]:
df[
    [
        "Appointment_Date",
        "Appointment_Month",
        "Appointment_Day",
        "Day_of_Week",
        "Is_Weekend"
    ]
].head()

,Appointment_Date,Appointment_Month,Appointment_Day,Day_of_Week,Is_Weekend
0,2026-01-01,1,1,3,0
1,2026-01-01,1,1,3,0
2,2026-01-01,1,1,3,0
3,2026-01-01,1,1,3,0
4,2026-01-01,1,1,3,0


In [25]:
df["Is_Weekend"]

0        0
1        0
2        0
3        0
4        0
        ..
11995    0
11996    0
11997    0
11998    0
11999    0
Name: Is_Weekend, Length: 12000, dtype: int64

In [26]:
df["Appointment_Hour"] = (
    pd.to_datetime(
        df["Appointment_Time"],
        format="%H:%M"
    ).dt.hour
)

In [27]:
print(
    df[
        ["Appointment_Time", "Appointment_Hour"]
    ].head()
)

  Appointment_Time  Appointment_Hour
0            08:00                 8
1            09:10                 9
2            09:30                 9
3            10:00                10
4            10:20                10


In [28]:
df["Peak_Hour"] = df["Appointment_Hour"].isin(
    [10, 11, 12, 13, 14, 15]
).astype(int)

In [29]:
appointment_minutes = (
    pd.to_datetime(
        df["Appointment_Time"],
        format="%H:%M"
    ).dt.hour * 60
    +
    pd.to_datetime(
        df["Appointment_Time"],
        format="%H:%M"
    ).dt.minute
)

arrival_minutes = (
    pd.to_datetime(
        df["Arrival_Time"],
        format="%H:%M"
    ).dt.hour * 60
    +
    pd.to_datetime(
        df["Arrival_Time"],
        format="%H:%M"
    ).dt.minute
)

df["Arrival_Delay_Minutes"] = (
    arrival_minutes - appointment_minutes
)

In [30]:
df[
    [
        "Appointment_Time",
        "Arrival_Time",
        "Arrival_Delay_Minutes"
    ]
].head(10)

,Appointment_Time,Arrival_Time,Arrival_Delay_Minutes
0,08:00,07:54,-6
1,09:10,09:33,23
2,09:30,09:46,16
3,10:00,09:54,-6
4,10:20,10:17,-3
5,10:20,10:45,25
6,11:10,11:21,11
7,11:10,11:11,1
8,11:50,11:55,5
9,11:50,12:11,21


In [31]:
print(df.columns.tolist())

['Patient_ID', 'Appointment_Date', 'Appointment_Time', 'Arrival_Time', 'Department', 'Doctor_ID', 'Patient_Type', 'Queue_Length', 'Doctors_Available', 'Appointments_Count', 'Consultation_Duration_Min', 'Waiting_Time_Minutes', 'Appointment_Month', 'Appointment_Day', 'Day_of_Week', 'Is_Weekend', 'Appointment_Hour', 'Peak_Hour', 'Arrival_Delay_Minutes']


In [32]:
df_encoded = pd.get_dummies(
    df,
    columns=["Department", "Patient_Type"],
    drop_first=True
)

print("Original Shape:", df.shape)
print("Encoded Shape:", df_encoded.shape)

Original Shape: (12000, 19)
Encoded Shape: (12000, 25)


In [33]:
print(df_encoded.columns.tolist())

['Patient_ID', 'Appointment_Date', 'Appointment_Time', 'Arrival_Time', 'Doctor_ID', 'Queue_Length', 'Doctors_Available', 'Appointments_Count', 'Consultation_Duration_Min', 'Waiting_Time_Minutes', 'Appointment_Month', 'Appointment_Day', 'Day_of_Week', 'Is_Weekend', 'Appointment_Hour', 'Peak_Hour', 'Arrival_Delay_Minutes', 'Department_Dermatology', 'Department_ENT', 'Department_General Medicine', 'Department_Gynecology', 'Department_Neurology', 'Department_Orthopedics', 'Department_Pediatrics', 'Patient_Type_Returning']


In [34]:
X = df_encoded.drop(
    columns=[
        "Patient_ID",
        "Appointment_Date",
        "Appointment_Time",
        "Arrival_Time",
        "Doctor_ID",
        "Waiting_Time_Minutes"
    ]
)

y = df_encoded["Waiting_Time_Minutes"]

print("X Shape:", X.shape)
print("y Shape:", y.shape)

X Shape: (12000, 19)
y Shape: (12000,)


In [35]:
print("Features used by the model:\n")

for feature in X.columns:
    print(feature)

Features used by the model:

Queue_Length
Doctors_Available
Appointments_Count
Consultation_Duration_Min
Appointment_Month
Appointment_Day
Day_of_Week
Is_Weekend
Appointment_Hour
Peak_Hour
Arrival_Delay_Minutes
Department_Dermatology
Department_ENT
Department_General Medicine
Department_Gynecology
Department_Neurology
Department_Orthopedics
Department_Pediatrics
Patient_Type_Returning


In [36]:
print("Missing values in X:")
print(X.isnull().sum().sum())

print("\nMissing values in y:")
print(y.isnull().sum())

Missing values in X:
0

Missing values in y:
0


In [37]:
processed_df = pd.concat(
    [X, y],
    axis=1
)

processed_df.to_csv(
    "../data/processed_hospital_data.csv",
    index=False
)

print("Processed dataset saved successfully!")

Processed dataset saved successfully!
